<a href="https://colab.research.google.com/github/KishoreKumar477/Qlora_on_facebook-opt-1.3b/blob/main/qlora_handson_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA Hands-On: 4-bit Quantization + LoRA


We'll switch to a bigger model than yesterday's DistilBERT (67M params) — a bigger model is where quantization actually matters. We'll use `facebook/opt-1.3b` (1.3B params), a causal language model, and fine-tune it lightly on a small instruction-style dataset.

**Recap from theory:**
- Frozen backbone → quantized to 4-bit NF4, never updated, just read during forward pass
- LoRA adapters (A×B) → kept in bf16, fully trainable, receive gradients normally
- A straight-through estimator lets gradients flow *through* the 4-bit frozen weights to reach the LoRA adapters, without ever updating the 4-bit weights themselves

## Cell 1: Install dependencies

`bitsandbytes` is the library that actually implements 4-bit quantization (NF4) and the straight-through estimator machinery under the hood.

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.1 MB/s eta 0:00:00


## Cell 2: Confirm GPU is available

`bitsandbytes` 4-bit quantization requires a CUDA GPU — it won't work on CPU. Make sure your Colab runtime is set to GPU (Runtime → Change runtime type → T4 GPU) before running this.

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

CUDA available: True
Device: Tesla T4


## Cell 3: Baseline — load the model in full precision, measure memory

Before quantizing anything, let's see what loading this model *normally* (bf16, no quantization) costs in GPU memory. This is our "before" number.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "facebook/opt-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_name)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_fp16 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")

mem_fp16 = torch.cuda.memory_allocated() / 1024**3
print(f"GPU memory after loading in bf16: {mem_fp16:.2f} GB")

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GPU memory after loading in bf16: 2.45 GB


**Write this number down** — for a 1.3B parameter model in bf16 (2 bytes/param), you should see roughly ~2.5–3 GB just to hold the weights. This is our baseline to compare against.

## Cell 4: Free the fp16 model before loading the quantized version

We need to clear GPU memory so the next measurement is clean and not polluted by the model we just loaded.

In [4]:
del model_fp16
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("Cleared. Current allocated memory:", torch.cuda.memory_allocated() / 1024**3, "GB")

Cleared. Current allocated memory: 0.0 GB


## Cell 5: Load the SAME model in 4-bit (QLoRA-style)

`BitsAndBytesConfig` is where the quantization actually gets specified:
- `load_in_4bit=True` — quantize weights to 4-bit on load
- `bnb_4bit_quant_type="nf4"` — use the NF4 format (tuned for neural network weight distributions, not generic 4-bit)
- `bnb_4bit_compute_dtype=torch.bfloat16` — when the frozen weights are dequantized on-the-fly for the forward/backward pass math, do that computation in bf16 (this is the straight-through estimator machinery from theory)
- `bnb_4bit_use_double_quant=True` — an extra trick: quantize the quantization constants themselves, squeezing out a bit more memory

In [5]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

mem_4bit = torch.cuda.memory_allocated() / 1024**3
print(f"GPU memory after loading in 4-bit: {mem_4bit:.2f} GB")
print(f"Reduction vs bf16: {mem_fp16 / mem_4bit:.2f}x smaller")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

GPU memory after loading in 4-bit: 0.78 GB
Reduction vs bf16: 3.14x smaller


**This is the payoff cell.** You should see the 4-bit load use roughly a quarter of the memory the bf16 load used — this is the actual, measured version of everything we discussed conceptually. This is the number that lets you fit a 7B, 13B, or even 70B model on hardware that couldn't hold it in full precision.

## Cell 6: Prepare the quantized model for training

Quantized models need a small preparation step before LoRA can be applied — this handles some numerical stability details (like casting layer norms back to fp32, and enabling gradient checkpointing) so training on top of 4-bit weights doesn't blow up.

In [6]:
from peft import prepare_model_for_kbit_training

model_4bit = prepare_model_for_kbit_training(model_4bit)

## Cell 7: Apply LoRA on top of the 4-bit model

Same idea as yesterday — freeze the (now 4-bit) backbone, add small trainable A×B matrices in bf16. Note `target_modules` here uses OPT's attention naming (`q_proj`, `v_proj`) instead of DistilBERT's (`q_lin`, `v_lin`) — different model, same underlying concept.

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

peft_model_qlora = get_peft_model(model_4bit, lora_config)
peft_model_qlora.print_trainable_parameters()

mem_after_lora = torch.cuda.memory_allocated() / 1024**3
print(f"GPU memory after adding LoRA adapters: {mem_after_lora:.2f} GB")

trainable params: 1,572,864 || all params: 1,317,330,944 || trainable%: 0.1194
GPU memory after adding LoRA adapters: 0.99 GB


Notice: adding LoRA barely increases memory at all, since the adapters are tiny compared to the 1.3B frozen backbone. Almost all your memory footprint is still just the 4-bit weights sitting there.

## Cell 8: Small instruction-style dataset

We'll use a tiny slice of an instruction-tuning dataset — this connects to yesterday's real-world discussion: fine-tuning here isn't teaching new facts, it's shaping response *style/format*, which is exactly what QLoRA is used for in practice.

In [8]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")
small_dataset = dataset.shuffle(seed=42).select(range(300))

def format_and_tokenize(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    tokenized = tokenizer(prompt, truncation=True, max_length=256, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

small_dataset = small_dataset.map(format_and_tokenize, remove_columns=small_dataset.column_names)
small_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Note: for causal LM fine-tuning, `labels` is just a copy of `input_ids` — the model is trained to predict the next token at every position, so the "target" is the same sequence, shifted internally by the model during loss computation.

## Cell 9: Train

Watch the GPU memory during this run (Colab's resource monitor, or add a print) — even while training a 1.3B model, memory should stay dramatically lower than a full-precision fine-tune of the same model would require.

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qlora-opt1.3b-alpaca",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    learning_rate=2e-4,
    save_strategy="no",
    report_to="none",
    bf16=True,
)

trainer = Trainer(
    model=peft_model_qlora,
    args=training_args,
    train_dataset=small_dataset,
)

trainer.train()

mem_peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"Peak GPU memory during training: {mem_peak:.2f} GB")

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.974410


Peak GPU memory during training: 1.99 GB


## Cell 10: Compare all the numbers side by side

This is the full picture — theory turned into measured reality.

In [12]:
print(f"bf16 model load (no training):        {mem_fp16:.2f} GB")
print(f"4-bit model load (no training):       {mem_4bit:.2f} GB")
print(f"4-bit + LoRA adapters (no training):  {mem_after_lora:.2f} GB")
print(f"Peak memory DURING training:          {mem_peak:.2f} GB")

bf16 model load (no training):        2.45 GB
4-bit model load (no training):       0.78 GB
4-bit + LoRA adapters (no training):  0.99 GB
Peak memory DURING training:          1.99 GB


NOtes can be implemented:
1. `use_dora=True` into the `LoraConfig` here — combine QLoRA + DoRA, the current 2026 production default